In [0]:
%run ./adls_auth


In [0]:
%run ./control_table

In [0]:
import json
from datetime import datetime, timedelta

SOURCE_NAME = "open_meteo_weather"
PROJECT_START_DATE = "2025-01-01"   # first day we want data for
PROJECT_END_DATE = "2025-12-31"     # last day we want data for (matches trips window)
CHUNK_DAYS = 30

In [0]:

default_watermark = (datetime.strptime(PROJECT_START_DATE, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
current_watermark = get_watermark(spark, SOURCE_NAME, default_watermark)

start_date = (datetime.strptime(current_watermark, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
tentative_end = datetime.strptime(start_date, "%Y-%m-%d") + timedelta(days=CHUNK_DAYS - 1)
project_end = datetime.strptime(PROJECT_END_DATE, "%Y-%m-%d")
end_date = min(tentative_end, project_end).strftime("%Y-%m-%d")

is_complete = start_date > PROJECT_END_DATE  # nothing left to process

result = {
    "start_date": start_date,
    "end_date": end_date,
    "is_complete": is_complete,
}

print(f"Watermark check: current={current_watermark}, next chunk={result}")
dbutils.notebook.exit(json.dumps(result))